# 05_Factor_Stability — 因子穩定性年度分析

## 目的

Phase 3-A 核心研究：找出在**所有年份都穩定有效**的因子。

**背景**：在 Phase 2 的 Fold-Internal IC 分析中，發現以下因子穩定性差異：

| 因子 | Fold 選中次數 (out of 24) | 狀態 |
|------|--------------------------|------|
| `vol_ratio` | 24/24 | 非常穩定 |
| `mom_1m_ra` | 23/24 | 非常穩定 |
| `trust_net_10d` | 22/24 | 穩定 |
| `mom_1m` | 21/24 | 穩定 |
| `price_52w` | **8/24** | **不穩定** |
| `foreign_net_20d` | **3/24** | **幾乎無效** |

本 Notebook 進一步按**年份**分解每個因子的 ICIR，找出：
- 方向每年一致的因子 → 可信賴的 alpha 信號
- 某年正/某年負的因子 → 噪音，不應依賴

## 設計原理（Single Source of Truth）

為了確保資料清理、過濾、流動性處理、因子計算 **完全等同於 strategy.py**，我們直接動態執行 `strategy.py` 的前半部，攔截其產生的 `X` 和 `y` 變數。不重複造輪子，保證 100% 同步。

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr
from tqdm import tqdm

print("Executing data generation pipeline from strategy.py (takes ~20 seconds)...")

# 儲存原本的工作路徑
original_cwd = os.getcwd()

try:
    # 將執行路徑強制切換回專案根目錄，這樣 strategy.py 裡面的 finmind_cache/ 相對路徑才找得到東西！
    if os.path.basename(original_cwd) == "Research":
        os.chdir("..")
        
    with open("strategy.py", "r", encoding="utf-8") as f:
        code = f.read()

    # 找到 Walk-Forward 的註解，切斷後面的模型訓練與回測邏輯
    cutoff = code.find("Walk-Forward Purged training")
    cutoff_line = code.rfind("\n", 0, cutoff)
    exe_code = code[:cutoff_line]

    env = {}
    exec(exe_code, env)

    X = env["X"]
    y = env["y"]
    test_dates = [d for d in env["all_dates"] if d >= env["TEST_START"]]
    print(f"\nPipeline finished! Extracted X shape: {X.shape}, test_dates: {len(test_dates)}")
finally:
    # 確保不管執行成不成功，都切換回原本的 Research 資料夾
    os.chdir(original_cwd)


Executing data generation pipeline from strategy.py (takes ~20 seconds)...
  Taiwan ML Stock Strategy  v4  (Optimized)

[1/7] Loading data...
  Loading OHLCV (for ATR & turnover)…


  OHLCV: 100%|██████████| 2511/2511 [00:07<00:00, 339.46it/s]


  Loading revenue…


  Revenue: 100%|██████████| 1931/1931 [00:03<00:00, 597.19it/s]


  Loading institution…


  Inst: 100%|██████████| 2407/2407 [00:15<00:00, 155.41it/s]


  close=(1737, 2511)  high=(1737, 2511)  rev=(87, 1931)

[2/7] Engineering factors…
  Total factors: 14

[3/7] Target & alignment…
  Dataset: X=(101018, 14), y=(101018,)
  Date range: 2020-01-31 ~ 2026-02-28
  Stocks: 1843 unique

[4/7] IC analysis…
                 IC Mean  IC Std    ICIR   IC>0%
trust_net_10d    -0.0182  0.0440 -0.4143  0.3243
atr_rel          -0.0711  0.1987 -0.3578  0.3514
mom_1m           -0.0370  0.1246 -0.2973  0.3919
rsi_14           -0.0278  0.1038 -0.2679  0.4054
mom_1m_ra        -0.0232  0.0948 -0.2447  0.4324
vol_ratio        -0.0228  0.0943 -0.2415  0.3919
rev_mom          -0.0110  0.0476 -0.2310  0.4189
price_52w         0.0339  0.1548  0.2188  0.6351
rev_accel        -0.0100  0.0500 -0.1991  0.4189
mom_6m           -0.0192  0.1213 -0.1582  0.4189
mom_3m           -0.0173  0.1323 -0.1311  0.5135
rev_yoy           0.0078  0.0788  0.0987  0.4730
breakout          0.0022  0.1504  0.0147  0.5270
foreign_net_20d   0.0005  0.0671  0.0073  0.5135

  [NOTE] 上表為全資

In [2]:
# ── 按年計算每個因子的月度 IC，整理成熱力圖格式
years = sorted(set(d.year for d in test_dates))
ic_by_year = {}

for factor in tqdm(X.columns, desc="Computing IC by year"):
    ic_by_year[factor] = {yr: [] for yr in years}
    for d in test_dates:
        try:
            if d in X.index.get_level_values(0):
                xf = X.xs(d, level=0)[factor].dropna()
                yf = y.xs(d, level=0).reindex(xf.index).dropna()
                xf = xf.reindex(yf.index)
                if len(yf) > 10:
                    ic_val, _ = spearmanr(xf.values, yf.values)
                    if not np.isnan(ic_val):
                        ic_by_year[factor][d.year].append(ic_val)
        except Exception:
            pass

print("IC computation done")

Computing IC by year: 100%|██████████| 14/14 [00:02<00:00,  5.41it/s]

IC computation done


In [3]:
# ── 彙整成 ICIR 熱力圖資料
icir_matrix = {}

for factor in X.columns:
    icir_matrix[factor] = {}
    for yr in years:
        ics = pd.Series(ic_by_year[factor][yr])
        if len(ics) >= 3:
            icir = ics.mean() / (ics.std() + 1e-8)
        else:
            icir = np.nan
        icir_matrix[factor][yr] = icir

icir_df = pd.DataFrame(icir_matrix).T

# 排序：依全期平均 |ICIR| 降序
overall_abs_icir = icir_df.abs().mean(axis=1).sort_values(ascending=False)
icir_df = icir_df.loc[overall_abs_icir.index]

print("ICIR Heatmap data (row=factor, col=year):")
display(icir_df.round(3))

ICIR Heatmap data (row=factor, col=year):


,2020,2021,2022,2023,2024,2025,2026
trust_net_10d,-0.322,-0.226,-0.515,-0.093,-0.658,-0.592,NaN
atr_rel,-0.061,-0.399,-0.330,-0.360,-0.794,-0.308,NaN
vol_ratio,-0.110,-0.607,-0.517,0.308,-0.445,0.023,NaN
mom_1m,-0.090,-0.557,-0.820,-0.054,-0.108,-0.314,NaN
price_52w,0.110,0.261,0.028,0.723,0.635,0.135,NaN
rev_mom,-0.192,-0.119,0.218,-0.313,-0.397,-0.544,NaN
mom_1m_ra,-0.163,-0.457,-0.828,0.084,-0.072,-0.096,NaN
rsi_14,-0.141,-0.155,-0.738,-0.234,-0.198,-0.151,NaN
rev_accel,-0.126,-0.176,0.183,-0.353,-0.333,-0.408,NaN
rev_yoy,0.204,-0.107,0.372,0.252,-0.164,0.142,NaN


In [4]:
# ── 視覺化：ICIR 年度熱力圖
fig = go.Figure(go.Heatmap(
    z           = icir_df.values,
    x           = [str(y) for y in icir_df.columns],
    y           = icir_df.index.tolist(),
    colorscale  = "RdBu",
    zmid        = 0,
    text        = icir_df.round(2).values,
    texttemplate= "%{text}",
    showscale   = True,
    colorbar    = dict(title="ICIR"),
))
fig.update_layout(
    title      = "Factor ICIR by Year (2020~2026) — Red=Bearish factor, Blue=Bullish factor",
    xaxis_title= "Year",
    yaxis_title= "Factor",
    template   = "plotly_dark",
    height     = 550,
)
fig.show()
print("閱讀指引：藍色=該年因子有效（負值因子用於空頭），紅色=反向，白色=無效")
print("理想的因子：每年都是同色（方向一致），不出現顏色在年份間切換")

閱讀指引：藍色=該年因子有效（負值因子用於空頭），紅色=反向，白色=無效
理想的因子：每年都是同色（方向一致），不出現顏色在年份間切換


In [5]:
# ── 因子穩定性評分：方向一致性（sign consistency）
stability_report = []
for factor in icir_df.index:
    vals = icir_df.loc[factor].dropna()
    if len(vals) == 0: continue
    pct_negative = (vals < 0).mean()
    pct_positive = (vals > 0).mean()
    sign_consistency = max(pct_negative, pct_positive)
    avg_abs_icir     = vals.abs().mean()
    direction        = "NEGATIVE (contrarian)" if pct_negative > pct_positive else "POSITIVE (momentum)"
    stability_report.append({
        "Factor"           : factor,
        "Direction"        : direction,
        "Sign Consistency" : f"{sign_consistency:.0%}",
        "Avg |ICIR|"       : f"{avg_abs_icir:.3f}",
        "Recommendation"   : "KEEP" if sign_consistency >= 0.80 else ("REVIEW" if sign_consistency >= 0.60 else "DROP"),
    })

stability_df = pd.DataFrame(stability_report).set_index("Factor")
print("Factor Stability Report:")
display(stability_df)

Factor Stability Report:


,Direction,Sign Consistency,Avg |ICIR|,Recommendation
Factor,,,,
trust_net_10d,NEGATIVE (contrarian),100%,0.401,KEEP
atr_rel,NEGATIVE (contrarian),100%,0.375,KEEP
vol_ratio,NEGATIVE (contrarian),67%,0.335,REVIEW
mom_1m,NEGATIVE (contrarian),100%,0.324,KEEP
price_52w,POSITIVE (momentum),100%,0.316,KEEP
rev_mom,NEGATIVE (contrarian),83%,0.297,KEEP
mom_1m_ra,NEGATIVE (contrarian),83%,0.283,KEEP
rsi_14,NEGATIVE (contrarian),100%,0.269,KEEP
rev_accel,NEGATIVE (contrarian),83%,0.263,KEEP


In [6]:
# ── ICIR 逐年折線圖（每個因子一條線）
keep_factors   = stability_df[stability_df["Recommendation"] == "KEEP"].index.tolist()
review_factors = stability_df[stability_df["Recommendation"] == "REVIEW"].index.tolist()
drop_factors   = stability_df[stability_df["Recommendation"] == "DROP"].index.tolist()

fig2 = go.Figure()
year_labels = [str(y) for y in icir_df.columns]

for factor in icir_df.index:
    rec = stability_df.loc[factor, "Recommendation"]
    color = {"KEEP": "#2962FF", "REVIEW": "#FFA000", "DROP": "#E53935"}[rec]
    dash  = {"KEEP": "solid", "REVIEW": "dash", "DROP": "dot"}[rec]
    fig2.add_trace(go.Scatter(
        x=year_labels, y=icir_df.loc[factor].values,
        name=f"{factor} ({rec})", mode="lines+markers",
        line=dict(color=color, dash=dash, width=2),
    ))

fig2.add_hline(y=0, line_color="white", line_dash="dash", line_width=1)
fig2.update_layout(
    title   = "Factor ICIR Trend by Year — Blue=KEEP, Orange=REVIEW, Red=DROP",
    xaxis_title="Year", yaxis_title="ICIR",
    template="plotly_dark", height=500,
    legend=dict(x=1.01, y=0.99),
)
fig2.show()

In [7]:
# ── 最終建議
print("="*60)
print("PHASE 3-A 結論：因子篩選建議")
print("="*60)
print(f"\n  KEEP  ({len(keep_factors)} 個）：{keep_factors}")
print(f"  REVIEW ({len(review_factors)} 個）：{review_factors}")
print(f"  DROP  ({len(drop_factors)} 個）：{drop_factors}")

print("\n下一步建議：")
print("  1. 在 strategy.py 中移除 DROP 的因子，只保留 KEEP + REVIEW 進入 Fold-Internal IC")
print("  2. 執行 strategy.py 觀察移除不穩定因子後的 CAGR 變化")
print("  3. 接著執行 07_Portfolio_Size_Sweep.ipynb（持股數量掃描）")

PHASE 3-A 結論：因子篩選建議

  KEEP  (10 個）：['trust_net_10d', 'atr_rel', 'mom_1m', 'price_52w', 'rev_mom', 'mom_1m_ra', 'rsi_14', 'rev_accel', 'mom_6m', 'mom_3m']
  REVIEW (3 個）：['vol_ratio', 'rev_yoy', 'foreign_net_20d']
  DROP  (1 個）：['breakout']

下一步建議：
  1. 在 strategy.py 中移除 DROP 的因子，只保留 KEEP + REVIEW 進入 Fold-Internal IC
  2. 執行 strategy.py 觀察移除不穩定因子後的 CAGR 變化
  3. 接著執行 07_Portfolio_Size_Sweep.ipynb（持股數量掃描）
